# Extract cells close to lost particle point

In [33]:
from typing import Iterable

In [8]:
import sys

from pathlib import Path

In [9]:
print("Python", sys.version, "at", sys.prefix)

Python 3.13.12 (main, Feb  3 2026, 22:52:12) [Clang 21.1.4 ] at /mnt/amarano-2tb/dev/mcnp/ep11/wrk/pc/pc11-case6/pc11-1.0/.venv


In [58]:
from mckit import Body, Universe
from mckit.body import Card
from mckit.parser import from_file

In [11]:
!ls

extract_cells_close_to_lp.ipynb  pc11-1.0  pc11-1.1-selected.i
extract_cells_close_to_lp.py	 pc11-1.1


In [12]:
Path.cwd()

PosixPath('/mnt/amarano-2tb/dev/mcnp/ep11/wrk/pc/pc11-case6/pc11-1.0')

In [13]:
cell_numbers_to_select = [187246]

In [14]:
!ls

extract_cells_close_to_lp.ipynb  pc11-1.0  pc11-1.1-selected.i
extract_cells_close_to_lp.py	 pc11-1.1


In [15]:
original_model_path = Path("pc11-1.1/pc11.i")
assert original_model_path.exists()

In [16]:
original_model = from_file(original_model_path)

In [17]:
original_universe = original_model.universe

In [18]:
len(original_universe)

43169

In [19]:
selected_cells: list[Body] = [c for c in original_universe if c.name() in cell_numbers_to_select]

In [20]:
len(selected_cells)

1

In [22]:
selected_surfaces = set()
for c in selected_cells:
    selected_surfaces.update(c._shape.get_surfaces())

In [55]:
len(selected_surfaces)

6

In [24]:
adjacent_cells: list[Body] = [c for c in original_universe if c.name() not in cell_numbers_to_select and c._shape.get_surfaces() & selected_surfaces]

In [25]:
len(adjacent_cells)

22

In [59]:
def list_names(cards: list[Card]) -> Iterable[int]:
    return (c.name() for c in cards)

In [60]:
sorted(list_names(adjacent_cells))

[186258,
 186259,
 187247,
 187248,
 187249,
 187296,
 187297,
 190182,
 190184,
 190185,
 190199,
 190200,
 190352,
 191652,
 191655,
 191657,
 191658,
 191918,
 191919,
 192228,
 192229,
 192243]

In [28]:
new_cells = sorted(selected_cells + adjacent_cells, key=Body.name)

In [61]:
list(list_names(new_cells))

[186258,
 186259,
 187246,
 187247,
 187248,
 187249,
 187296,
 187297,
 190182,
 190184,
 190185,
 190199,
 190200,
 190352,
 191652,
 191655,
 191657,
 191658,
 191918,
 191919,
 192228,
 192229,
 192243]

In [39]:
len(new_cells)

23

In [41]:
selected_universe = Universe(new_cells)

In [42]:
selected_universe.save("pc11-1.1/lp/selected.i")

In [44]:
!ls pc11-1.1/lp

comin			    lp-cell-fails-count.csv  selected.i
extract_lost_particles.log  lp-coordinates.csv
lost-particles.sqlite	    lp-details.txt


In [62]:
sorted(list_names(selected_surfaces))

[191560, 191652, 194422, 194446, 201368, 201392]

In [67]:
clashed_surface_numbers: set[int] = set([201392])

In [68]:
cells_with_clashed_surfaces: list[Body] = [c for c in original_universe if set(s.name() for s in c._shape.get_surfaces()) & clashed_surface_numbers]

In [66]:
sorted(list_names(cells_with_clashed_surfaces))

[186259,
 186313,
 187247,
 187297,
 187298,
 190182,
 190184,
 190185,
 190199,
 190200,
 191652,
 191655,
 191657,
 191658,
 192243]